## Data Preprocessing Phase

### Importing libraries

In [8]:
import tensorflow as tf
import numpy as np 
import pandas as pd

### Defining standard parameters for MobileNetV2

In [9]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'dataset/FMD_DATASET'

In [10]:
print("Loading Training Data:")


train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=0.2,
    subset='training',
    seed=42
)

Loading Training Data:
Found 14536 files belonging to 3 classes.
Using 11629 files for training.


In [11]:
print("\nLoading Validation Data:")

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, # Points to the path where your subfolders are stored
    shuffle=True, # Randomizes the order of the images during loading so the model does not learn based on folder/file ordering.
    batch_size=BATCH_SIZE, 
    image_size=IMG_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=42 # makes randomness predictable
)


Loading Validation Data:
Found 14536 files belonging to 3 classes.
Using 2907 files for validation.


In [12]:
class_names = train_dataset.class_names
print(f"\nSuccessfully mapped {len(class_names)} classes: {class_names}")


Successfully mapped 3 classes: ['incorrect_mask', 'with_mask', 'without_mask']


## Part 2 - Building the MobileNetV2 Architecture

### Importing Libraries

In [13]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models

### Loading pre-trained MobileNetV2 without its top classification layer 

In [14]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,  # Excluding the default ImageNet 1000-class classifier
    weights='imagenet'  # Use pre-trained ImageNet weights
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


### Freezzing the base model layers

In [15]:
base_model.trainable = False

### Building the full model pipeline

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3)) # must be exactly 224 pixels wide, 224 pixels tall, with 3 color channels (Red, Green, Blue)

### Applying MobileNetV2-specific preprocessing (scales pixel values to [-1, 1])

In [ ]:
x = preprocess_input(inputs) # instantly changes all the picture pixels between -1 and 1

### Passing through pre-trained base model

In [18]:
x = base_model(x, training=False)

### Pooling & Classification Head

In [19]:
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)  # Regularization to prevent overfitting
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs, outputs)

### Compiling the model

In [20]:
# We use sparse_categorical_crossentropy because class labels are integer-encoded
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Training the Model


### Defining the numbers of epochs

In [22]:
EPOCHS = 10

### Training the model


In [23]:
print("Starting model training...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)
print("\nTraining complete!")

Starting model training...
Epoch 1/10


/Users/rudra/Desktop/ML/ml_env2/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


364/364 ━━━━━━━━━━━━━━━━━━━━ 73s 195ms/step - accuracy: 0.9466 - loss: 0.1604 - val_accuracy: 0.9708 - val_loss: 0.0911
Epoch 2/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 76s 209ms/step - accuracy: 0.9696 - loss: 0.0902 - val_accuracy: 0.9763 - val_loss: 0.0757
Epoch 3/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 81s 221ms/step - accuracy: 0.9777 - loss: 0.0696 - val_accuracy: 0.9766 - val_loss: 0.0733
Epoch 4/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 100s 275ms/step - accuracy: 0.9792 - loss: 0.0634 - val_accuracy: 0.9776 - val_loss: 0.0701
Epoch 5/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 98s 268ms/step - accuracy: 0.9810 - loss: 0.0593 - val_accuracy: 0.9770 - val_loss: 0.0718
Epoch 6/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 99s 271ms/step - accuracy: 0.9818 - loss: 0.0533 - val_accuracy: 0.9770 - val_loss: 0.0756
Epoch 7/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 92s 252ms/step - accuracy: 0.9829 - loss: 0.0478 - val_accuracy: 0.9773 - val_loss: 0.0698
Epoch 8/10
364/364 ━━━━━━━━━━━━━━━━━━━━ 93s 255ms/step - accuracy: 0.9848 - loss: 0.0442 - va